# Goodreads Data Analysis
## Part 1: Loading and Cleaning with Pandas

Read in the `goodreads.csv` file, examine the data, and do any necessary data cleaning.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
pd.set_option('display.width', 500)
pd.set_option('display.max_columns', 100)

### Cleaning: Reading in the data
We read in and clean the data from `goodreads.csv`.

In [ ]:
# Read the data into a dataframe
df = pd.read_csv("data/goodreads.csv")

# Examine the first few rows of the dataframe
df.head()

We are missing the column names. We need to add these in.

`["rating", 'review_count', 'isbn', 'booktype','author_url', 'year', 'genre_urls', 'dir','rating_count', 'name']`

In [ ]:
# Load the dataframe properly with column names
df = pd.read_csv("data/goodreads.csv", header=None,
                 names=["rating", 'review_count', 'isbn', 'booktype','author_url',
                        'year', 'genre_urls', 'dir','rating_count', 'name'])

# Examine the first few rows of the dataframe
df.head()

### Cleaning: Examining the dataframe - quick checks

In [ ]:
# Check the types of the columns
df.dtypes

**Answer:** The columns `rating`, `year`, `rating_count`, and `review_count` are stored as `object` (string) instead of numeric. This happens because the raw CSV contains missing or invalid values that prevent pandas from inferring a numeric type. These columns will need cleaning before numeric analysis.

In [ ]:
print(df.shape)
df.columns

### Cleaning: Examining the dataframe - a deeper look

In [ ]:
# Get a sense of how many missing values there are in the dataframe
np.sum([df.rating.isnull()])

In [ ]:
# Try to locate where the missing values occur
df[df.rating.isnull()]

### Cleaning: Dealing with Missing Values

Rows with missing or invalid `year` values are not useful for a book-level analysis, so we drop them.

In [ ]:
df[df.year.isnull()]

In [ ]:
# Treat the missing or invalid values in your dataframe
df = df[df.year.notnull()]

In [ ]:
df.dtypes

In [ ]:
print(np.sum(df.year.isnull()))
print(np.sum(df.rating_count.isnull())) 
print(np.sum(df.review_count.isnull())) 
# We removed seven rows
df.shape

### Convert columns to int

In [ ]:
df.rating_count = df.rating_count.astype(int)
df.review_count = df.review_count.astype(int)
df.year = df.year.astype(int)

In [ ]:
df.dtypes

In [ ]:
df.loc[df.genre_urls.isnull(), 'genre_urls'] = ""
df.loc[df.isbn.isnull(), 'isbn'] = ""

## Part 2: Parsing and Completing the Data Frame

In [ ]:
# Get the first author_url
test_string = df.author_url[0]
test_string

In [ ]:
# Test out some string operations to isolate the author name
test_string.split('/')[-1].split('.')[1:][0]

In [ ]:
# Write a function that accepts an author url and returns the author's name
def get_author(url):
    name = url.split('/')[-1].split('.')[1:][0]
    return name

In [ ]:
# Apply the get_author function to the 'author_url' column using '.map' 
# and add a new column 'author' to store the names
df['author'] = df.author_url.map(get_author)
df.author[0:5]

In [ ]:
df.genre_urls.head()

In [ ]:
# Examine some examples of genre_urls
# Test out some string operations to isolate the genre name
test_genre_string = df.genre_urls[0]
genres = test_genre_string.split('|')
for e in genres:
    print(e.split('/')[-1])
    "|".join(genres)

In [ ]:
def split_and_join_genres(url):
    genres = url.split('|')
    genres = [e.split('/')[-1] for e in genres]
    return "|".join(genres)

In [ ]:
split_and_join_genres("/genres/young-adult|/genres/science-fiction")

In [ ]:
split_and_join_genres("")

In [ ]:
df['genres'] = df.genre_urls.map(split_and_join_genres)
df.head()

In [ ]:
df[df.author == "Marguerite_Yourcenar"]

In [ ]:
del df['genre_urls']

In [ ]:
df.to_csv("data/cleaned-goodreads.csv", index=False, header=True)

## Part 3: Grouping

In [ ]:
# Observations with negative years (books written before Common Era)
df[df.year < 0].head()
# These are books written before the Common Era (BCE, equivalent to BC).

In [ ]:
dfgb_author = df.groupby('author')
type(dfgb_author)

In [ ]:
dfgb_author.count()

In [ ]:
dfgb_author['author'].count()

In [ ]:
dfgb_author[['rating', 'rating_count', 'review_count', 'year']].describe()

In [ ]:
ratingdict = {}
for author, subset in dfgb_author:
    ratingdict[author] = (subset['rating'].mean(), subset['rating'].std())
ratingdict

In [ ]:
# Get the best-rated book(s) for every year in our dataframe
for year, subset in df.groupby('year'):
    # Find the best book of the year
    bestbook = subset[subset.rating == subset.rating.max()]
    if bestbook.shape[0] > 1:
        print(year, bestbook.name.values, bestbook.rating.values)
    else:
        print(year, bestbook.name.values[0], bestbook.rating.values[0])

## Train/Test Split and Linear Regression

In [ ]:
from sklearn.model_selection import train_test_split

X = df[['rating']]
y = df['rating_count']

# make 90 percent of data as the training set and the rest as test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)
print('Training rows:', len(X_train))
print('Test rows:', len(X_test))
print('Train mean rating:', round(X_train['rating'].mean(), 3))
print('Test mean rating:', round(X_test['rating'].mean(), 3))
print('Train target mean:', round(y_train.mean(), 3))
print('Test target mean:', round(y_test.mean(), 3))

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1. Create the model and fit it on the training data only
linreg = LinearRegression()
linreg.fit(X_train, y_train)

# 2. Predict on both the training and the test set
y_pred_train = linreg.predict(X_train)
y_pred_test = linreg.predict(X_test)

# 3. Inspect the learned straight line: y = slope * rating + intercept
print("Coefficient (slope):", linreg.coef_[0])
print("Intercept:", linreg.intercept_)

# 4. Evaluate on both sets to check for overfitting
print("Train R^2:", round(r2_score(y_train, y_pred_train), 3))
print("Test R^2:", round(r2_score(y_test, y_pred_test), 3))
print("Train MSE:", round(mean_squared_error(y_train, y_pred_train), 3))
print("Test MSE:", round(mean_squared_error(y_test, y_pred_test), 3))

# 5. Plot the fitted line against the actual test data
plt.figure(figsize=(8, 5))
plt.scatter(X_test, y_test, alpha=0.4, label="Actual")
plt.plot(X_test, y_pred_test, color="red", linewidth=2, label="Fitted line")
plt.xlabel("Rating")
plt.ylabel("Rating count")
plt.title("Linear regression: rating_count vs rating (test set)")
plt.legend()
plt.show()